# Task 2 — Conditional GAN (CGAN)

**Goal:** Build a CGAN that uses class labels as conditional input to generate
images. This introduces the core idea of *conditional* GANs — the Generator and
Discriminator both receive the label alongside the usual noise/image input, so
the network learns to produce class-appropriate images rather than random ones.

**Scope:** deliberately kept simple and fast to train (per the assignment brief,
which uses "square"/"circle"-style basic shape generation as its own example).
We condition on the **Oxford-102 flower class label** (0–101) rather than full
text embeddings — label-conditioning is the foundational CGAN concept; text-level
conditioning via CLIP embeddings comes later in Tasks 5 and 6.

**Design choices (carried over from Task 4's fixes):**
- Dataset downloaded to **local disk**, not read repeatedly from Drive
- Images downsized to **64×64** to keep training fast and stable on a free-tier T4
- Checkpoints and sample grids saved locally, then copied to Drive once at the end

## 1. Setup

In [ ]:
!pip install -q torch torchvision matplotlib

import os
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision.datasets import Flowers102
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 2. Load Oxford-102 to local disk

Same reliable pattern as Task 4: download to `/content/data`, never read
repeatedly over a Drive network mount during training.

In [ ]:
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)

IMG_SIZE = 64
NUM_CLASSES = 102

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),  # scale to [-1, 1], matches Tanh generator output
])

train_set = Flowers102(root=DATA_ROOT, split='train', download=True, transform=transform)
val_set   = Flowers102(root=DATA_ROOT, split='val', download=True, transform=transform)

# Combine train+val for more training data — we don't need a held-out test set
# for a generative model in this context (no classification accuracy to evaluate)
full_train = torch.utils.data.ConcatDataset([train_set, val_set])
print(f'Total training images: {len(full_train)}')

BATCH_SIZE = 64
dataloader = DataLoader(full_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

## 3. Visualize a batch to confirm data looks right

In [ ]:
sample_batch, sample_labels = next(iter(dataloader))
grid = make_grid(sample_batch[:16], nrow=4, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0))
plt.axis('off')
plt.title('Sample training images (64x64)')
plt.savefig('/content/sample_batch.png', dpi=150)
plt.show()

## 4. Define the Generator and Discriminator

**Conditioning mechanism:** the class label is passed through an `nn.Embedding`
layer and concatenated with the noise vector (Generator) or with the image's
spatial feature maps (Discriminator), following the standard CGAN approach.

In [ ]:
LATENT_DIM = 100
LABEL_EMBED_DIM = 50

class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES, embed_dim=LABEL_EMBED_DIM, img_size=IMG_SIZE):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, embed_dim)
        input_dim = latent_dim + embed_dim

        self.init_size = img_size // 16  # 64 / 16 = 4
        self.fc = nn.Linear(input_dim, 256 * self.init_size * self.init_size)

        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(256),
            nn.Upsample(scale_factor=2),                              # 4 -> 8
            nn.Conv2d(256, 128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Upsample(scale_factor=2),                              # 8 -> 16
            nn.Conv2d(128, 64, 3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Upsample(scale_factor=2),                              # 16 -> 32
            nn.Conv2d(64, 32, 3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Upsample(scale_factor=2),                              # 32 -> 64
            nn.Conv2d(32, 3, 3, stride=1, padding=1),
            nn.Tanh(),
        )

    def forward(self, noise, labels):
        label_emb = self.label_embed(labels)
        x = torch.cat([noise, label_emb], dim=1)
        x = self.fc(x)
        x = x.view(x.size(0), 256, self.init_size, self.init_size)
        img = self.conv_blocks(x)
        return img


class Discriminator(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, img_size=IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        # Embed label into a full extra image channel, concatenated with the RGB image
        self.label_embed = nn.Embedding(num_classes, img_size * img_size)

        def block(in_ch, out_ch, bn=True):
            layers = [nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1)]
            if bn:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(4, 32, bn=False),   # 64 -> 32   (3 image channels + 1 label channel)
            *block(32, 64),            # 32 -> 16
            *block(64, 128),           # 16 -> 8
            *block(128, 256),          # 8  -> 4
        )
        self.adv_layer = nn.Sequential(
            nn.Linear(256 * (img_size // 16) ** 2, 1),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        label_map = self.label_embed(labels).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([img, label_map], dim=1)
        x = self.model(x)
        x = x.view(x.size(0), -1)
        validity = self.adv_layer(x)
        return validity


generator = Generator().to(device)
discriminator = Discriminator().to(device)

print(generator)
print(discriminator)

## 5. Training setup

In [ ]:
adversarial_loss = nn.BCELoss()

lr = 0.0002
beta1 = 0.5
optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

N_EPOCHS = 100          # adjust down if you're short on Colab time
SAMPLE_EVERY = 10       # save a sample grid every N epochs

os.makedirs('/content/outputs', exist_ok=True)
os.makedirs('/content/checkpoints', exist_ok=True)

# Fixed noise + fixed labels so we can visually track the SAME classes improving over epochs
FIXED_CLASSES = [0, 25, 50, 72, 90, 101]  # a spread of flower classes
n_fixed = len(FIXED_CLASSES)
fixed_noise = torch.randn(n_fixed, LATENT_DIM, device=device)
fixed_labels = torch.tensor(FIXED_CLASSES, device=device)

## 6. Training loop

Standard CGAN adversarial training: the Discriminator learns to distinguish
real (image, label) pairs from fake ones; the Generator learns to fool it.

**Time estimate:** on a T4 GPU, ~100 epochs over ~7,000 images at batch size 64
typically takes roughly 30–45 minutes. Reduce `N_EPOCHS` above if you need it
to run faster — CGANs on simple label conditioning usually show visible
progress well before 100 epochs.

In [ ]:
g_losses, d_losses = [], []

for epoch in range(1, N_EPOCHS + 1):
    epoch_g_loss, epoch_d_loss = 0.0, 0.0

    for real_imgs, labels in dataloader:
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)
        bs = real_imgs.size(0)

        valid = torch.ones(bs, 1, device=device)
        fake = torch.zeros(bs, 1, device=device)

        # ---- Train Generator ----
        optimizer_G.zero_grad()
        noise = torch.randn(bs, LATENT_DIM, device=device)
        gen_labels = torch.randint(0, NUM_CLASSES, (bs,), device=device)
        gen_imgs = generator(noise, gen_labels)
        g_loss = adversarial_loss(discriminator(gen_imgs, gen_labels), valid)
        g_loss.backward()
        optimizer_G.step()

        # ---- Train Discriminator ----
        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(real_imgs, labels), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach(), gen_labels), fake)
        d_loss = (real_loss + fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()

    avg_g = epoch_g_loss / len(dataloader)
    avg_d = epoch_d_loss / len(dataloader)
    g_losses.append(avg_g)
    d_losses.append(avg_d)

    print(f'Epoch [{epoch}/{N_EPOCHS}]  D_loss: {avg_d:.4f}  G_loss: {avg_g:.4f}')

    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        generator.eval()
        with torch.no_grad():
            samples = generator(fixed_noise, fixed_labels)
        grid = make_grid(samples, nrow=n_fixed, normalize=True)
        save_image(grid, f'/content/outputs/epoch_{epoch:03d}.png')
        generator.train()

# Save final model weights locally
torch.save(generator.state_dict(), '/content/checkpoints/generator.pth')
torch.save(discriminator.state_dict(), '/content/checkpoints/discriminator.pth')
print('\nTraining complete. Checkpoints and sample grids saved locally.')

## 7. Plot training curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(g_losses, label='Generator loss')
plt.plot(d_losses, label='Discriminator loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CGAN Training Losses')
plt.legend()
plt.tight_layout()
plt.savefig('/content/outputs/training_curves.png', dpi=150)
plt.show()

## 8. Compare early vs late generated samples

This is the key visual evidence that the CGAN learned something — same fixed
classes and noise, shown at an early epoch vs a late epoch.

In [ ]:
from PIL import Image as PILImage

early_path = '/content/outputs/epoch_001.png'
late_path = f'/content/outputs/epoch_{N_EPOCHS:03d}.png' if N_EPOCHS % SAMPLE_EVERY == 0 else None

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].imshow(PILImage.open(early_path))
axes[0].set_title('Epoch 1 (early training)')
axes[0].axis('off')

if late_path and os.path.exists(late_path):
    axes[1].imshow(PILImage.open(late_path))
    axes[1].set_title(f'Epoch {N_EPOCHS} (final)')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('/content/outputs/early_vs_late_comparison.png', dpi=150)
plt.show()

## 9. Generate new samples for specific classes on demand

A quick demo function — pass in any class index (0–101) and get generated
flower images for it. This is effectively a mini version of what Task 6 will
wrap into a full text-driven pipeline.

In [ ]:
def generate_for_class(class_idx, n_samples=4):
    generator.eval()
    with torch.no_grad():
        noise = torch.randn(n_samples, LATENT_DIM, device=device)
        labels = torch.full((n_samples,), class_idx, device=device, dtype=torch.long)
        imgs = generator(noise, labels)
    grid = make_grid(imgs, nrow=n_samples, normalize=True)
    plt.figure(figsize=(10, 3))
    plt.imshow(grid.permute(1, 2, 0).cpu())
    plt.axis('off')
    plt.title(f'Generated samples for class {class_idx}')
    plt.show()
    generator.train()

# Try a few classes
generate_for_class(72)   # e.g. rose (verify actual index against CLASS_NAMES from Task 4 if needed)
generate_for_class(50)

## 10. Copy results to Drive

Same one-time-copy pattern as before: mount, copy the small set of output
files, done. No repeated Drive I/O during training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/elevance-skills/task2_cgan'
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copytree('/content/outputs', os.path.join(DRIVE_DIR, 'outputs'), dirs_exist_ok=True)
shutil.copytree('/content/checkpoints', os.path.join(DRIVE_DIR, 'checkpoints'), dirs_exist_ok=True)

print('Copied outputs and checkpoints to:', DRIVE_DIR)

## 11. Summary of findings

Fill this in after training, then copy into `NOTES.md` and today's daily log:
- How did loss curves behave (stable, oscillating, one network dominating)?
- Did generated samples visibly improve from early to late epochs?
- Are shapes/colors class-appropriate, or still fairly noisy/abstract?
- What would you try next to improve quality (more epochs, different architecture, etc.)?